# 02 - Preprocessing

Notebook skeleton aligned with the preprocessing stage described in `README.md`.

## Goals

- load raw market data from `data/raw`
- normalize the target asset
- create sliding windows
- prepare train, validation, and test splits


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_ROOT = PROJECT_ROOT / "data" / "raw"
EXTRACTION_DATE = "2026-04-19"
TARGET_SYMBOL = "NVDA"
LOOKBACK = 60


In [ ]:
target_path = RAW_ROOT / "market_data" / "source=yfinance" / f"symbol={TARGET_SYMBOL}" / f"extraction_date={EXTRACTION_DATE}" / "ohlcv.csv"
target_df = pd.read_csv(target_path, parse_dates=["date"])
target_df = target_df[["date", "close"]].dropna().copy()
target_df.head()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_target = scaler.fit_transform(target_df[["close"]])
scaled_target[:5]

In [ ]:
def create_sequences(data: np.ndarray, lookback: int):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_target, LOOKBACK)
X = X.reshape(X.shape[0], X.shape[1], 1)
X.shape, y.shape

In [ ]:
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

X_train.shape, X_val.shape, X_test.shape

## Preprocessing Runbook

Use this section as the starting point for preprocessing.

Current status of the project:
- there is no dedicated preprocessing CLI yet
- preprocessing currently starts after raw generation
- this notebook is the active entrypoint for preprocessing analysis

### CLI commands used before this notebook

```bash
# 1. Generate the raw zone locally
python scripts/generate_raw.py --skip-s3

# 2. Generate the raw zone locally and upload parquet files to S3
python scripts/generate_raw.py

# 3. Open the preprocessing notebook
jupyter lab notebooks/02_preprocessing.ipynb
# or
jupyter notebook notebooks/02_preprocessing.ipynb
```

### Important input values for preprocessing

- `RAW_ROOT`: local source folder, default `data/raw`
- `EXTRACTION_DATE`: partition used to locate the raw snapshot
- `TARGET_SYMBOL`: asset used for the baseline preprocessing flow, default `NVDA`
- `LOOKBACK`: sliding window size, default `60`
- target feature: `close`
- split proportions: train `70%`, validation `15%`, test `15%`

### Input file conventions

- local preprocessing reads from `data/raw/market_data/source=yfinance/symbol=<SYMBOL>/extraction_date=<DATE>/ohlcv.csv`
- S3 storage keeps the same partition logic, but uses `.parquet`
- minimum columns expected in this notebook: `date`, `close`

### Recommended checks at the start of preprocessing

1. confirm the raw generation completed successfully
2. confirm the chosen `EXTRACTION_DATE` exists under `data/raw`
3. confirm the selected symbol file contains `date` and `close`
4. confirm there are no missing values before sequence creation
5. persist the scaler later alongside the trained model
